In [18]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
from notebooks.imports import *

import scipy.io as sio
import h5py

import hdf5storage

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Load Configs

In [19]:
from config import dir_config, main_config

raw_dir = Path(dir_config.data.raw)
compiled_dir = Path(dir_config.data.compiled)


### Utils functions

In [20]:
def determine_choice(row):
    if row['is_valid']:
        if row['outcome']:
            return row['target']
        else:
            return 'left' if row['target'] == 'right' else 'right'
    else:
        return np.nan

def get_prior_condition(df):
    valid_df = df[df['is_valid']].copy()  # Ensure valid_df is a copy to avoid SettingWithCopyWarning

    # Calculate trial counts and percentages for each condition within valid trials
    condition_counts = valid_df.groupby(['target', 'color']).size().reset_index(name='counts')
    total_counts = condition_counts.groupby('color')['counts'].transform('sum')
    condition_counts['percentage'] = (condition_counts['counts'] / total_counts) * 100

    # Filter conditions meeting the 60% criterion
    conditions_met = condition_counts[(condition_counts['percentage'] > 60)].copy()  # Make a copy to safely modify

    # Prepare the output based on conditions met
    if not conditions_met.empty:
        # Use .loc to modify 'condition' column safely
        conditions_met.loc[:, 'condition'] = conditions_met.apply(lambda x: 'gr' if x['target'] == "right" and x['color'] == "green"
                                                                else ('gl' if x['target'] == "left" and x['color'] == "green"
                                                                        else ('rr' if x['target'] == "right" and x['color'] == "red"
                                                                            else 'rl')), axis=1)
        return conditions_met[['condition', 'target', 'color']].values.tolist()[0]
    else:
        return ['eq', -1, -1]

### Raw Data Column Description
#### Codes
    - 1001              Start trial
    - 2500              Fixation point ON
    - 2000              Targets appears (white choice cue for correct)
    - 2009              Distractor appears (white choice cue for wrong)
    - 4000:4001     Target (correct choice) is left (4000) or right (4001)
    - 4100:4199     Difficulty levels or coherence levels (4100= easiest)
    - 5000              Glass pattern appears
    - 5500              Glass pattern disappears
    - 5001              (invalid trial)	Failed to hold fixation
    - 5004              (invalid trial)	Failed to hold target
    - 5005              (invalid trial)	Anticipatory saccade
    - 5006              Chose distractor (wrong choice)
    - 5007              Failed to respond on time
    - 5510              Correct choice
    - 1503              The actual key press
    - 6101:6102     Glass pattern color (6101: green)
##### Events (starting with one)
    - 3rd column: GP orientation 
    - 4th column: % coherence {4100:100; 4101:35; 4102:13; 4103:0}
    - 5th column: GP color
    - 9th column: correct? 
##### Time (starting with one)
    - Reaction Time: 8th column - 7th column

## Compiling data from all subjects

In [21]:
reject_subject = main_config['moca_rejection']

In [22]:
subject_metadata = pd.read_csv(raw_dir / 'session_metadata_detailed_all_data.csv', encoding='latin1')
subjects_from_metadata = subject_metadata['subject_id'].unique()

# get all .mat files from the raw data directory
compiled_mat_files = list(Path(compiled_dir).glob("*.mat"))
subjects_from_data_file = [f.stem.split("_")[0] for f in compiled_mat_files]

In [23]:
ucla = ["CG", "COH", "MBY", "DP", "FUR", "LBR", "MAR", "SMI", "PAM", "RW", "BBK", "BER", "DCAM", "ALE", "DMO", "AJL", "SKU"]
case_western = ["RBA", "RDE", "SGA", "LHO", "RSH", "RZA", "SNO"]

stanford = subject_metadata.loc[
    subject_metadata["subject_id"].str.match(r"^P\d+$", na=False)  # Ensure it starts with 'P' followed by numbers
    & (subject_metadata["subject_id"].str.extract(r"P(\d+)")[0].astype(float) <= 24),  # Extract number and compare
    "subject_id",
].unique()
harvard = subject_metadata.loc[
    subject_metadata["subject_id"].str.match(r"^P\d+$", na=False)  # Ensure it starts with 'P' followed by numbers
    & (subject_metadata["subject_id"].str.extract(r"P(\d+)")[0].astype(float) > 24),  # Extract number and compare
    "subject_id",
].unique()

harvard_hc = subject_metadata.loc[subject_metadata["subject_id"].str.match(r"^HC\d+$", na=False), "subject_id"].unique()

# go through the subject_metadata and update the site
for subject_id in subject_metadata["subject_id"]:
    if subject_id in ucla:
        site = "UCLA"
    elif subject_id in case_western:
        site = "Case_Western"
    elif subject_id in stanford:
        site = "Stanford"
    elif subject_id in harvard or subject_id in harvard_hc:
        site = "Harvard"
    else:
        site = "Unknown"
    subject_metadata.loc[subject_metadata["subject_id"] == subject_id, "experiment_site"] = site


In [24]:
# remove rejected subjects from subjects_from_metadata and subjects_from_data_file
subjects_from_metadata = [s for s in subjects_from_metadata if s not in reject_subject]
subjects_from_data_file = [s for s in subjects_from_data_file if s not in reject_subject]

In [25]:

print(f"Metadata contains extra: {set(subjects_from_metadata) - set(subjects_from_data_file)}")
print(f"Data files contain extra: {set(subjects_from_data_file) - set(subjects_from_metadata)}")
print(len(set(subjects_from_data_file)), len(set(subjects_from_metadata)))

# Only assert if *all* subject IDs follow the pattern (none outside it)
if len(subjects_from_metadata) == len(subjects_from_metadata) and len(subjects_from_data_file) == len(subjects_from_data_file):
	assert len(set(subjects_from_data_file)) == len(set(subjects_from_metadata)), "Mismatch between metadata and data files"
else:
	pass


Metadata contains extra: set()
Data files contain extra: set()
60 60


In [26]:
aggregate_df_list = []  # Use a list to collect DataFrames

for session_file in compiled_mat_files:
	session_data = hdf5storage.loadmat(str(session_file))

	df = pd.DataFrame(
		{
			"color": np.select([session_data["event"][:, 4] == 6101, session_data["event"][:, 4] == 6102], ["green", "red"], default=None),
			"coherence": np.select([session_data["event"][:, 3] == 4100, session_data["event"][:, 3] == 4101, session_data["event"][:, 3] == 4102, session_data["event"][:, 3] == 4103], [100, 35, 13, 0], default=np.nan),
			"target": np.select([session_data["event"][:, 2] == 4000, session_data["event"][:, 2] == 4001], ["left", "right"], default=None),
		}
	)

	invalid_trials = np.sort(np.where((session_data["event"][:, 8] == 5007) | (session_data["event"][:, 7] == 5005) | (session_data["event"][:, 7] == 0) | (session_data["event"][:, 7] == 5008))[0])
	df["is_valid"] = True
	df.loc[invalid_trials, "is_valid"] = False

	df["outcome"] = np.nan
	df.loc[np.where(session_data["event"][:, 8] == 5510)[0], "outcome"] = 1
	df.loc[np.where(session_data["event"][:, 7] == 5006)[0], "outcome"] = 0

	df["choice"] = df.apply(determine_choice, axis=1)
	df["reaction_time"] = session_data["time"][:, 7] - session_data["time"][:, 6]

	df["prior"], df["prior_direction"], df["prior_color"] = get_prior_condition(df)

	df["subject_id"] = session_file.name.split("_")[0]
	if df["subject_id"].iloc[0].startswith("HC"):
		df["group"] = "hc"
		df["medication"] = "None"
	else:
		df["group"] = "pd"
		df["medication"] = session_file.name.split("_")[-2]
		df["medication"] = df["medication"].apply(lambda x: x[:-4].lower())

	df["session_filename"] = session_file.name

	aggregate_df_list.append(df)  # Append DataFrame to the list

# Concatenate all DataFrames in the list at once
aggregate_df = pd.concat(aggregate_df_list, ignore_index=True)

# replace empty strings with NaN
aggregate_df.replace('', np.nan, inplace=True)


In [27]:
aggregate_df['subject_id'].unique()

array(['DP', 'HC6', 'P33', 'DMO', 'P27', 'PAM', 'COH', 'P1', 'P11', 'HC5',
       'P10', 'BER', 'P25', 'P13', 'P19', 'CG', 'BBK', 'RZA', 'FUR',
       'LBR', 'P34', 'P29', 'RBA', 'P6', 'P23', 'DCAM', 'HC8', 'P32',
       'P31', 'P20', 'P2', 'P12', 'LHO', 'P30', 'MBY', 'RDE', 'P18',
       'SKU', 'AJL', 'P3', 'P24', 'HC7', 'ALE', 'HC9', 'P7', 'P4', 'SGA',
       'HC2', 'MAR', 'P9', 'P8', 'RSH', 'P16', 'HC13', 'P15', 'P26', 'RW',
       'HC1', 'SMI', 'P28', 'HC12', 'P22', 'P17', 'SNO', 'HC3', 'P14'],
      dtype=object)

In [28]:
# rearrange columns
aggregate_df = aggregate_df[['subject_id', 'group', 'medication', 'prior', 'prior_direction', 'prior_color', 'color', 'coherence', 'target', 'is_valid', 'outcome', 'choice', 'reaction_time', "session_filename"]]

In [29]:
aggregate_df.to_csv(Path(compiled_dir, 'aggregate_all_data.csv'), index=False)

subject_metadata.to_csv(Path(compiled_dir, 'aggregate_all_metadata.csv'), index=False)